# Meta-MedRAG — Ablation Study

Evalue la contribution de chaque module en le desactivant.

> **Prerequisite:** A100 + Meta_MedRAG_A100_Final.ipynb executed.

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────
import sys, os
from pathlib import Path

from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

REPO = Path('/content/meta_medrag')
if not REPO.exists():
    os.system('git clone https://github.com/nourboudhina/meta-medrag.git /content/meta_medrag')

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Setup complete')

In [ ]:
# ── Cell 2: Run ablation — IU-Xray VQA (50 samples) ─────────
print('Running ablation study on IU-Xray VQA...')
!python scripts/run_ablation.py \
    --condition all \
    --dataset iu_xray \
    --task vqa \
    --limit 50 \
    --config configs/config.yaml
print('Ablation done')

In [ ]:
# ── Cell 3: Run ablation — IU-Xray Report Generation ─────────
print('Running ablation study on IU-Xray report generation...')
!python scripts/run_ablation.py \
    --condition all \
    --dataset iu_xray \
    --task report_gen \
    --limit 50 \
    --config configs/config.yaml

In [ ]:
# ── Cell 4: Display results + generate LaTeX Table 4.4 ───────
import json, shutil
from pathlib import Path

results_vqa = json.load(open('experiments/ablations/iu_xray_vqa_ablation.json'))
results_rg  = json.load(open('experiments/ablations/iu_xray_report_gen_ablation.json'))

print('=' * 70)
print('  ABLATION RESULTS — IU-Xray VQA')
print('=' * 70)
print(f'  {"Condition":<18} | {"Accuracy":>9} | {"F1":>9}')
print('  ' + '-' * 42)
for cond, m in results_vqa.items():
    marker = ' ← FULL' if cond == 'full' else ''
    print(f'  {cond:<18} | {m.get("accuracy",0):>9.4f} | {m.get("f1",0):>9.4f}{marker}')

print()
print('  ABLATION RESULTS — IU-Xray Report Generation')
print('=' * 70)
print(f'  {"Condition":<18} | {"BLEU-4":>9} | {"ROUGE-L":>9} | {"METEOR":>9}')
print('  ' + '-' * 58)
for cond, m in results_rg.items():
    print(f'  {cond:<18} | {m.get("bleu4",0):>9.4f} | {m.get("rouge_l_f1",0):>9.4f} | {m.get("meteor",0):>9.4f}')

In [ ]:
# ── Cell 5: Generate LaTeX Table 4.4 ─────────────────────────
lines = []
lines.append(r'\begin{table}[ht]')
lines.append(r'\centering')
lines.append(r'\caption{Ablation study results on IU-Xray. Each row disables one module.')
lines.append(r'\textbf{Full} = complete Meta-MedRAG system.}')
lines.append(r'\label{tab:ablation}')
lines.append(r'\begin{tabular}{lcccc}')
lines.append(r'\toprule')
lines.append(r'\textbf{Condition} & \textbf{Accuracy} & \textbf{F1} & \textbf{BLEU-4} & \textbf{ROUGE-L} \\')
lines.append(r'\midrule')

for cond in ['full','no_probe','no_dpo','no_filter','baseline']:
    vqa = results_vqa.get(cond, {})
    rg  = results_rg.get(cond, {})
    acc = vqa.get('accuracy', 0)
    f1  = vqa.get('f1', 0)
    bl  = rg.get('bleu4', 0)
    rl  = rg.get('rouge_l_f1', 0)
    label = r'\textbf{' + cond + '}' if cond == 'full' else cond
    lines.append(f'{label} & {acc:.4f} & {f1:.4f} & {bl:.4f} & {rl:.4f} \\\\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

latex = '\n'.join(lines)
print(latex)

out = Path('experiments/results/table_4_4_ablation.tex')
out.write_text(latex)

import shutil
drive_res = Path('/content/drive/MyDrive/meta_medrag_results/results')
drive_res.mkdir(parents=True, exist_ok=True)
shutil.copy(out, drive_res/'table_4_4_ablation.tex')
print(f'\nSaved: {out}\nDrive backup OK')